# Hedge Design Dashboard — SPX tail hedge

Load a book and design changes to it: editor, sizing, strike selection, roll and monetization planning, and stress testing.

In [ ]:
# Imports
import contextlib
import datetime
from datetime import datetime as dt

import matplotlib.pyplot as plt
from IPython.display import display

from deltadewa.analysis import (
    PortfolioAnalyzer,
    ScenarioGridCache,
    get_volatility_stats,
)
from deltadewa.analysis.roll_status import (
    compute_moneyness_drift,
    estimate_roll_up_cost,
)
from deltadewa.constants import OptionType
from deltadewa.dashboard import (
    ChangeLogDisplay,
    MonteCarloStalenessWidget,
    StressDashboard,
    VolatilityProfileDisplay,
    start_session,
)
from deltadewa.marketdata import MarketDataError
from deltadewa.reporting import PortfolioChangeTracker
from deltadewa.visualization import OptionCharts
from deltadewa.widgets import (
    HedgeHealthDashboard,
    NetHedgeSummary,
    PortfolioWidgets,
)

In [ ]:
# Dashboard session: portfolio, market data provider, IPS policy,
# and GlobalAssumptions — all bootstrapped in one call.
# Offline-safe by default; pass use_live_market_data=True for live
# CBOE/FRED data.
ctx = start_session(
    role="design",
    globals_dict=globals(),
    auto_load_default=False,
                    )

portfolio = ctx.portfolio
ips_config = ctx.ips_config
dashboard_config = ctx.dashboard_config
market_data = ctx.market_data
global_assumptions = ctx.global_assumptions
reporter = ctx.reporter
portfolio_changelog = ctx.changelog
portfolio_serializer = ctx.serializer
EXPORT_DIR = ctx.export_dir
today = ctx.today

portfolio_widgets = PortfolioWidgets(
    portfolio, portfolio_serializer, portfolio_changelog,
)

vol_stats = get_volatility_stats(portfolio)

reporter.success("Setup complete.")

## Load / Import book

In [ ]:
# Import Widget and Portfolio Change Tracker

# Create tracker — seeds the baseline snapshot from the current portfolio state
position_tracker = PortfolioChangeTracker(
    portfolio=portfolio,
    logger=portfolio_changelog,
    reporter=reporter,
)

# Import widget — reset the tracker after a successful import so it
# doesn't diff against the pre-import snapshot
import_widget = portfolio_widgets.display_import(
    on_import_success=position_tracker.reset,
)
display(import_widget)

## Assumptions (editable scenario inputs)

In [ ]:
display(global_assumptions.display())

## Position Editor

In [ ]:
# Interactive Position Editor using widgets module

# Pass the callback to the position editor widget
position_editor = portfolio_widgets.create_position_editor(
    on_change_callback=position_tracker.as_callback(),
)

display(position_editor)

print("\n💡 All position changes will be automatically logged")

## Net Summary (what I'm building)

In [ ]:
# Hedge Summary
net_hedge_summary = NetHedgeSummary(portfolio)
display(net_hedge_summary.display())

## Hedge Health

In [ ]:
# Display Hedge Health Dashboard
health_dashboard = HedgeHealthDashboard(portfolio, config=dashboard_config)
dashboard_loader = health_dashboard.display_config_loader()
display(dashboard_loader)
display(health_dashboard.display())
# Update when portfolio changes
health_dashboard.update()

## Sizing workbench  [NEW — not yet built]

In [ ]:
# [NEW — not yet built] Sizing workbench
#
# Intended inputs (handbook §3525 "Portfolio Hedge Sizing
# Framework", §2499 "Beta-Adjusted Hedge Sizing"):
#   - ips_config.drawdown.max_tolerance_pct  (drawdown tolerance ->
#     required offset)
#   - ips_config.convexity.crash_scenario_pct / target_min_pct /
#     target_max_pct  (notional via crash payoff ratio)
#   - ips_config.budget.annual_carry_pct  (carry check)
#
# 5-step flow: drawdown tolerance -> required offset -> notional via
# crash payoff ratio -> carry check.
pass  # noqa: PIE790

## Strike ladder builder  [NEW — not yet built]

In [ ]:
# [NEW — not yet built] Strike ladder builder
#
# Intended: delta-based strike selection across a ladder of
# maturities (handbook §2642 "Strike Selection", §2764
# "Delta-Based Strike Selection", §2801 "Maturity Selection").
pass  # noqa: PIE790

## Roll planner

In [ ]:
# Roll Planner — read-only preview of candidate roll-up strikes.
# Uses the same building blocks as analysis/roll_status.py,
# without mutating the portfolio: compute_moneyness_drift() for
# current vs. entry %OTM, and estimate_roll_up_cost() for the
# cash cost of moving to a new strike.
current_spot = portfolio.spot_price
with contextlib.suppress(MarketDataError):
    current_spot = market_data.get_spot(portfolio.get_symbol())

long_puts = [
    p for p in portfolio.positions
    if p.option.option_type == OptionType.PUT and p.quantity > 0
]

if not long_puts:
    print("No long puts to evaluate.")
else:
    for position in long_puts:
        moneyness = compute_moneyness_drift(position, current_spot)
        print(
            f"PUT ${position.option.strike_price:.0f} x{position.quantity} | "
            f"OTM now: {moneyness.current_otm_pct:+.1f}%",
        )
        for new_strike in (
            position.option.strike_price * 0.95,
            position.option.strike_price * 0.90,
        ):
            cost = estimate_roll_up_cost(
                position, new_strike, position.option.volatility,
            )
            print(f"    -> ${new_strike:.0f}: roll-up cost ${cost:,.2f}")

## Monetization planner  [NEW — not yet built]

In [ ]:
# [NEW — not yet built] Monetization planner
#
# Intended: evaluate ips_config.monetization.schedule (staged
# sell_pct at each gain_pct trigger) against vol-spike / drawdown /
# hedge-value triggers (handbook Part VIII §3729 "Typical
# Monetization Triggers").
pass  # noqa: PIE790

## Monte Carlo (run for risk/reward)

In [ ]:
# Monte Carlo Simulation - Run Now For Use in Downstream Analysis
#
# This cell runs the Monte Carlo simulation once and stores results on the
# portfolio object. All downstream widgets and visualizations will use
# these cached results for consistency and efficiency.

num_simulations = int(global_assumptions.monte_carlo_num_sims.value)
include_underlying = global_assumptions.monte_carlo_inc_ul.value

reporter.subheader(
    "Monte Carlo Simulation - Run Now For Use in Downstream Analysis",
    )
if len(portfolio.positions) > 0:
    print(
        f"Running {num_simulations:,} simulations with underlying: "
        f"{include_underlying}...",
        )
    mc_results = portfolio.run_monte_carlo_simulation(
        num_simulations=num_simulations,
        include_underlying=include_underlying,
        )

    # Mark Monte Carlo results as fresh (not stale)
    portfolio.monte_carlo_stale = False
    portfolio.monte_carlo_timestamp = dt.now(tz=datetime.UTC)

    reporter.success(
        f"Simulation complete: "
        f"{mc_results['num_simulations']:,} valid scenarios",
    )
    reporter.info(
        f"We have {'stale' if portfolio.monte_carlo_stale else 'fresh'} results"
        f" as of {portfolio.monte_carlo_timestamp}",
    )
else:
    reporter.error("No positions - Monte Carlo simulation skipped")
    mc_results = None

reporter.divider()

In [ ]:
# Initialize Scenario Grid Cache and Stress Dashboard

# This caches expensive scenario calculations for better performance
scenario_cache = ScenarioGridCache(max_size=128)
reporter.success("Scenario cache initialized")

analyzer = PortfolioAnalyzer(portfolio)
reporter.success("Portfolio analyzer ready")

stress_dashboard = StressDashboard(
    portfolio=portfolio,
    analyzer=analyzer,
    cache=scenario_cache,
    global_assumptions=global_assumptions,
    reporter=reporter,
)
reporter.success("StressDashboard ready")

## P&L of proposed structure

In [ ]:
# P&L Distribution Chart with Annotated Key Metrics
if len(portfolio.positions) > 0:
    charts = OptionCharts(portfolio)

    # Generate the chart
    fig = charts.plot_pnl_distribution_with_metrics(
        spot_range_pct=max(25.0, global_assumptions.spot_shock_pct.value * 100),
        num_points=500,  # High resolution for smooth curve
        include_underlying=True,  # Include underlying position
        show_probability_overlay=True,  # Show probability density function
    )

    plt.show()

else:
    print("No positions to analyze. Add positions in BUILD mode.")

## Stress: Time×Price

In [ ]:
# Time vs Price Heatmap Analysis
if len(portfolio.positions) > 0:
    time_heatmap_widget = stress_dashboard.create_time_heatmap(metric="pnl")
    display(time_heatmap_widget)
else:
    reporter.error("No positions to analyze. "
                   "Add positions in BUILD mode first.")

## Stress: Vol×Price

In [ ]:
# Interactive Stress Test Heatmap (Spot x Volatility)
if len(portfolio.positions) > 0:
    spot_vol_widget = stress_dashboard.create_spot_vol_heatmap(
        metric="pnl", days_forward=0,
        )
    display(spot_vol_widget)
else:
    reporter.error("No positions to analyze. "
                   "Add positions in BUILD mode first.")

## Risk / Reward (Monte Carlo)

In [ ]:
# Monte Carlo Staleness Check & Re-run Widget
mc_widget = MonteCarloStalenessWidget(
    portfolio, num_simulations, include_underlying, reporter,
)
is_stale = mc_widget.check_and_warn()

reporter.header(" Monte Carlo P&L Distribution Analysis")

mc_results = portfolio.monte_carlo_results
if mc_results is None:
    reporter.warning("No Monte Carlo results found. Running simulation now...")
    mc_results = portfolio.run_monte_carlo_simulation(
        num_simulations=num_simulations,
        include_underlying=include_underlying,
    )
    portfolio.monte_carlo_stale = False
    portfolio.monte_carlo_timestamp = dt.now(tz=datetime.UTC)
else:
    if not is_stale:
        reporter.success(
            "Using cached Monte Carlo results from earlier simulation",
            )
    else:
        reporter.warning("Using STALE cached results (portfolio was modified)")

# Delegate all rendering to StressDashboard
stress_dashboard.display_risk_reward_summary(mc_results)

## Volatility Profile

In [ ]:
# Display Portfolio Volatility Profile
VolatilityProfileDisplay(portfolio, reporter).display(vol_stats)

## Session Change Log

In [ ]:
# Portfolio Change Log Display
ChangeLogDisplay(portfolio_changelog, reporter).display()

## Export working portfolio

In [ ]:
# Final Export Widget
final_export_widget = portfolio_widgets.display_export()
display(final_export_widget)

reporter.header("SESSION COMPLETE")
print("Remember to export your portfolio to save your work!")
print("Use the widget above to export in JSON, CSV, or YAML format.")
reporter.divider()